# 02. VRFB GraphRAG: Performance & Failure Analysis Q&A

This notebook demonstrates a proof-of-concept (PoC) GraphRAG system for VRFB.

**Workflow:**
1. Retrieve context from Neo4j
2. Inject the domain-specific graph context into the LLM prompt
3. Generate answers using LLM based on causal relationships from the context

## Setup and Connections

In [ ]:
# Environment Setup and Imports
%pip install -q -r requirements.txt

from neo4j import GraphDatabase
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load Credentials
load_dotenv()
URI = os.getenv('NEO4J_URI')
USER = os.getenv('NEO4J_USER')
PASSWORD = os.getenv('NEO4J_PASSWORD')
BASE_URL = os.getenv('LLM_URL')
API_KEY = os.getenv('LLM_API_KEY')

# Connect to database
AUTH = (USER, PASSWORD)
driver = GraphDatabase.driver(URI, auth=AUTH)

# LLM connection
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

## Graph Retrieval

In [4]:
def retrieve_all_knowledge():
    ''' Retrieve all nodes and relationships from the database '''

    query = """
    MATCH (n)
    OPTIONAL MATCH (n)-[r]->(m)
    RETURN n.name AS start_node, type(r) AS relation_type, m.name AS end_node
    """
    
    context = []
        
    with driver.session() as session:
        result = session.run(query)
        
        for record in result:
            # components with connections
            if record["relation_type"] and record["end_node"]:
                line = f"[{record['start_node']}] --({record['relation_type']})--> [{record['end_node']}]"
                context.append(line)
            # stand alone points without connections
            elif record["start_node"]:
                line = f"[{record['start_node']}]"
                context.append(line)
                
    # remove duplicates and combine all the text
    unique_lines = list(set(context))
    context_str = "The following is the complete knowledge graph data of vanadium redox flow battery:\n" + "\n".join(unique_lines)
    
    return context_str

print(retrieve_all_knowledge())

The following is the complete knowledge graph data of vanadium redox flow battery:
[Anolyte Aged Charge State] --(CONTAINS)--> [V3+]
[Carbon Corrosion] --(CAUSES)--> [Flow Blockage]
[Battery Management System]
[Separator] --(IS_A)--> [Membrane & Separator]
[V5+] --(REDUCED_TO)--> [V4+]
[V4+] --(OXIDIZED_TO)--> [V5+]
[Exothermic Reaction] --(SPEEDS_UP)--> [V5+ Precipitation]
[Anolyte Aged Oxidation State] --(RESTORED_TO)--> [Anolyte Fresh Oxidation State]
[Low Membrane Thickness] --(CAUSES)--> [High Crossover]
[Catholyte Aged State] --(HAS_CONDITION)--> [Catholyte Aged Charge State]
[High Voltage Efficiency] --(HAS_IMPACT_ON)--> [Round-Trip Energy Efficiency Trade-off]
[Membrane Burst] --(CAUSES)--> [Electrolyte Mixing]
[High Flow Resistance] --(CAUSES)--> [High Pump Power Required]
[Flow Battery System] --(CONTAINS)--> [Pipe]
[V5+] --(CAUSES)--> [V5+ Precipitation]
[V3+] --(REDUCED_TO)--> [V2+]
[Temperature Sensor]
[Short Circuit] --(CAUSES)--> [Safety Issue]
[Flow Battery System] --(C

## Q&A and Evaluation

In [5]:
def ask_battery_expert(user_question):
    '''  Ask the battery expert LLM a question based on the knowledge graph context. '''
    
    graph_context = retrieve_all_knowledge()
    
    # LLM should only use the graph context to answer the question
    prompt = f"""You are a vanadium redox flow battery expert, and you have the following knowledge graph context:{graph_context}. Please answer the following question BASED ON THE KNOWLEDGE GRAPH CONTEXT. You don't have to show the user the original nodes and relationships, just give your conclusions. If the question is not related to the knowledge graph context, please answer "I don't know". User question: {user_question}
    """
    
    # Call LLM
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

# test
question = 'What might be the root cause of high internal resistance and what will happen if the internal resistance is too high?'
print(ask_battery_expert(question))

**Possible root causes of high internal resistance (based on the knowledge graph):**  
- Electrode masking  
- Low flowrate  
- Excessive membrane thickness  
- Carbon corrosion  

**What happens if internal resistance becomes too high:**  
High internal resistance leads to low voltage efficiency, which in turn degrades the round‑trip energy efficiency (i.e., the battery delivers less usable energy for the same input).
